# imports

In [1]:
# !git clone https://github.com/moryacho/street-pattern-classifier.git

# !pip install huggingface_hub geopandas osmnx
# !pip install torch_geometric
# !pip install "folium>=0.12" matplotlib mapclassify

# !pip uninstall torch torchvision torchaudio -y
# !pip cache purge
# !pip install torch torchvision torchaudio

import torch

In [2]:
import geopandas as gpd
import numpy as np
import osmnx as ox
import pandas as pd
import time
from tqdm import tqdm
import os
import networkx as nx
import json

import matplotlib.pyplot as plt
from collections import Counter
from scipy import stats
import warnings

from shapely.geometry import LineString, Polygon, Point
import shapely
import pickle
import ast



# cities 

## names

In [9]:
osm_city_names = [
    # East Asia
    "Beijing, China",
    "Chengdu, China",
    "Chongqing, China",
    "Guangzhou, China",
    "Hangzhou, China",
    "Shanghai, China",
    "Shenzhen, China",
    "Wuhan, China",
    "Nanjing, China",
    "Taipei, Taiwan",
    "Tokyo, Japan",
    "Osaka, Japan",
    "Seoul, South Korea",
    
    # South Asia
    "Bangkok, Thailand",
    "Delhi, India",
    "Singapore, Singapore",
    "Kuala Lumpur, Malaysia",
    # "Mumbai, India",
    "Hyderabad, India",
    "Kathmandu, Nepal",
    
    # Oceania
    "Melbourne, Australia",
    "Sydney, Australia",
    "Auckland, New Zealand",
    "Brisbane, Australia",
    
    # Africa
    "Cairo, Egypt",
    "Casablanca, Morocco",
    "Lagos, Nigeria",
    "Nairobi, Kenya",
    # "Johannesburg, South Africa",
    
    # Europe
    "Amsterdam, Netherlands",
    "Brussels, Belgium",
    "London, United Kingdom",
    "Paris, France",
    "Berlin, Germany",
    "Munich, Germany",
    "Madrid, Spain",
    "Milan, Italy",
    "Moscow, Russia",
    "Stockholm, Sweden",
    "Bucharest, Romania",
    "Saint Petersburg, Russia",
    
    # North America
    "New York City, USA",  # Важно: именно "New York City", а не просто "New York"
    "Los Angeles, USA",
    "Detroit, USA",
    "Philadelphia, USA",
    "San Antonio, USA",
    "San Diego, USA",
    "Toronto, Canada",
    "Montreal, Canada",
    "Chicago, USA",
    "Houston, USA",
    "Phoenix, USA",
    
    # South America
    "Lima, Peru",
    "Bogota, Colombia",
    "Sao Paulo, Brazil",  # Можно также "São Paulo, Brazil"
    "Caracas, Venezuela",
    "Brasilia, Brazil",    # Можно также "Brasília, Brazil"
    "Asuncion, Paraguay"
]

## centroids and buffers

In [12]:
BUFFER = 20000
data = []

for city in tqdm(osm_city_names):
    gdf = ox.geocode_to_gdf(city)
    if gdf.crs != 4326:
        print(f"Исходная CRS: {gdf.crs}")
    gdf_projected = gdf.to_crs(gdf.estimate_utm_crs())
    center_point_projected = gdf_projected.centroid.iloc[0]
    
    point_gdf_utm = gpd.GeoDataFrame(geometry=[center_point_projected], crs=gdf_projected.crs)
    buffer_utm = point_gdf_utm.buffer(BUFFER)
    buffer_geo = buffer_utm.to_crs(gdf.crs)    
    center_point_geo = gpd.GeoDataFrame(geometry=[center_point_projected], crs=gdf_projected.crs).to_crs(gdf.crs).geometry.iloc[0]
    
    data.append({
        "city_name": city,
        "centroid": center_point_geo,
        "geometry": buffer_geo.geometry.iloc[0]
    })

100%|██████████| 56/56 [00:05<00:00, 11.17it/s]


In [13]:
gdf_centroids_polygons = gpd.GeoDataFrame(data, crs='EPSG:4326', geometry='geometry')

## drive graph data

In [20]:
# Source - https://stackoverflow.com/a/77965352
# Posted by Nick ODell
# Retrieved 2026-03-19, License - CC BY-SA 4.0

import osmnx as ox

# Must be an overpass instance which supports attic
ox.settings.overpass_endpoint = "https://overpass-api.de/api"

def query_year(coordinate, year):
    date = f'{year}-01-01T00:00:00Z'
    # Request attic data
    ox.settings.overpass_settings = '[out:json][timeout:{timeout}][date:"' + date + '"]{maxsize}'
    graph = ox.graph.graph_from_polygon(coordinate)
    # Restore old settings
    ox.settings.overpass_settings = '[out:json][timeout:{timeout}]{maxsize}'
    return graph


In [21]:
gdf_centroids_polygons['graph_2026'] = None
gdf_centroids_polygons['graph_2020'] = None


graphs = []
for i in tqdm(range(len(gdf_centroids_polygons))):
    row = gdf_centroids_polygons.iloc[0]
    polygon = row['geometry']

    drive_graph_2026 = ox.graph_from_polygon(polygon, network_type='drive')
    drive_graph_2020 = query_year(polygon, 2020)
    
    gdf_centroids_polygons.at[row.name, 'graph_2026'] = drive_graph_2026
    gdf_centroids_polygons.at[row.name, 'graph_2020'] = drive_graph_2020


  7%|▋         | 4/56 [03:15<42:23, 48.92s/it]  


KeyboardInterrupt: 